# 主题
B站 IA 虚拟歌姬音乐作品播放量与传播分析

## 核心分析问题
- 哪些原作者的作品最受欢迎（播放量排行）？
- 播放量与点赞、收藏、弹幕等指标的相关性如何？
- IA音乐作品随时间推移的热度趋势与"过气"分析：以发布时间为横轴，聚合各时间段内作品的当前总/平均播放量，观察整体热度走向；可按原作者、是否初投稿等条件筛选样本空间，判断特定作者或IA歌姬是否"过气"
- 原唱 vs 翻唱作品的表现差异
- 不同标签/风格的歌曲播放表现对比

## 项目五大功能模块
1. **文件上传兼容**：支持CSV/Excel/JSON格式上传，前端注明支持格式
2. **数据预览与筛选**：上传后展示表格，支持排序与筛选
3. **数据清洗**：解析标题提取歌名/创作者/歌姬，处理缺失值，构造衍生特征
4. **数据分析**：描述统计、相关性分析、趋势分析、排行榜
5. **数据可视化**：ECharts交互式图表（折线、柱状、散点、饼图）
6. **AI问答助手**：接入DeepSeek V4 Flash，自然语言问答驱动数据分析

## 前期准备（非项目功能）
- 编写B站爬虫 → 采集 ≥1200 条IA音乐视频数据 → 产出 `data/ia_music_data.csv`

## 开发计划
- [V] 编写B站爬虫，获取数据（前期准备）
- [ ] 搭建Flask项目骨架与目录结构
- [ ] 实现5大核心后端模块
- [ ] 编写前端页面模板
- [ ] 接入DeepSeek V4 Flash API
- [ ] 集成测试与调试
- [ ] 准备演示材料（PPT、视频、README）

# 项目架构

## 目录结构
```
project/
├── app.py                     # Flask 主入口，注册路由与启动
├── config.py                  # 配置文件（密钥、路径、模型参数）
├── requirements.txt           # Python 依赖清单
├── train_classifier.py        # QLoRA 微调脚本（独立运行）
├── data/
│   ├── ia_music_data.csv      # 内置数据集（爬虫产出，≥2000行）
│   └── labeled_samples.json   # 人工标注样本（30-50条，用于QLoRA微调）
├── models/
│   └── content_classifier_lora/  # QLoRA 微调后的适配器权重
├── modules/                   # 五大核心后端模块
│   ├── __init__.py
│   ├── file_reader.py         # 模块1：文件读取与格式兼容
│   ├── preprocessor.py        # 模块2：数据预处理与清洗
│   ├── content_classifier.py  # 模块2子模块：QLoRA分类器加载与推理
│   ├── analyzer.py            # 模块3：数据分析引擎
│   ├── visualizer.py          # 模块4：可视化JSON生成
│   └── ai_assistant.py        # 模块5：DeepSeek AI问答助手
├── static/
│   ├── css/
│   │   └── style.css          # 全局样式
│   └── js/
│       └── main.js            # 前端交互逻辑 + ECharts 图表渲染
├── templates/
│   ├── base.html              # 公共布局模板（导航栏、页脚）
│   ├── index.html             # 首页：文件上传 + 数据预览
│   ├── analysis.html          # 分析页：数据清洗 + 统计 + 可视化
│   └── chat.html              # 问答页：AI 对话式数据分析
└── uploads/                   # 用户上传文件临时存储
```

## 前期准备（独立脚本，非项目模块）
```
crawler/
├── bilibili_crawler.py        # B站数据爬虫
├── test_crawler.py            # 爬虫验证脚本
└── preview_tags.py            # 标签统计与筛选工具
```

## 整体数据流
```
[前期准备] 爬虫 → CSV + 人工标注 → labeled_samples.json
                                          ↓
                              train_classifier.py → models/ (QLoRA)
                                          ↓
用户上传/内置数据 → [模块1: file_reader] → pandas DataFrame (视频粒度)
                              ↓
      [模块2: preprocessor] → 去重 → QLoRA分类 → 标题解析 → 衍生特征
                              ↓                         ↓
                    df_clean (视频粒度)          df_songs (歌曲粒度)
                              ↓                         ↓
      [模块3: analyzer] ← 两个DataFrame，分别分析
                              ↓
      [模块4: visualizer] → ECharts option JSON → 前端渲染图表
                              ↓
用户自然语言提问 → [模块5: ai_assistant] ← 数据上下文注入
                              ↓
                  DeepSeek V4 Flash API
                              ↓
                  回答 + 图表数据 → 前端展示
```

## 页面与模块对应关系
| 页面 | 路由 | 涉及的模块 |
|------|------|-----------|
| index.html | `/` | 模块1 (file_reader) |
| analysis.html | `/analysis` | 模块2 + 模块3 + 模块4 |
| chat.html | `/chat` | 模块5 (ai_assistant) |

## 技术选型理由
- **Flask**：轻量Python Web框架，适合课程项目规模，学习曲线平缓
- **pandas**：数据处理核心，课堂重点内容，功能满足需求
- **ECharts**：前端JS图表库，交互性强（缩放、悬停详情、动态切换），演示效果优于Matplotlib静态图
- **Qwen2.5-1.5B + QLoRA**：小模型微调，Mac本地可跑，展示AI能力（课程加分）
- **DeepSeek V4 Flash**：性价比高的大模型API，中文理解能力强，适合数据分析问答场景

# 数据来源：B站爬虫（前期准备）

> 爬虫是独立的前期准备工作，不属于项目的五大功能模块。运行一次产出 CSV 后即可供 Flask 应用使用。

## 爬取策略
使用 B站 Web API（无需登录即可访问的公开接口）搜索 IA 相关音乐视频：

1. **搜索接口**：`https://api.bilibili.com/x/web-interface/search/type`
   - 关键词：`IA`、`IA 原创曲`、`IA 翻唱`、`IA 虚拟歌姬` 等
   - 分区限定：音乐区（rid=3）
   - 排序方式：播放量降序 / 发布时间降序
   - 分页获取，每页50条，总计爬取 ≥ 1200 条（防止清洗后不足1000）

2. **视频详情接口**（补充统计信息）：`https://api.bilibili.com/x/web-interface/view`
   - 获取更详细的互动数据（硬币数、分享数等）

3. **爬取字段**：
   - 基础信息：aid, bvid, title, author(UP主名), publish_date, duration
   - 互动数据：play_count, danmaku_count, comment_count, favorite_count, coin_count, share_count, like_count
   - 分类信息：tags, category, url

4. **反爬措施**：
   - 请求间隔 0.5~1s，避免触发频率限制
   - 设置 User-Agent 模拟浏览器
   - 异常重试机制（最多3次）

5. **输出**：`data/ia_music_data.csv`，UTF-8编码

## 增量爬取去重策略

多关键词搜索会产生交叉结果（同一条视频可能同时匹配"IA 原创曲"和"IA 虚拟歌姬"），多次运行爬虫也会重复遇到已爬视频。以 `bvid` 为主键去重：

```
1. 若 data/ia_music_data.csv 已存在 → 读取全部 bvid → set
2. 爬取过程中每条结果：
   - bvid not in set → 写入 CSV，set.add(bvid)
   - bvid in set → 跳过
3. 爬取结束后输出统计：新增 X 条，跳过 Y 条
```

> 设计决策：对已存在的 bvid 选择"跳过"而非"更新"，保持数据一致性，避免同一次分析中同一视频出现新老两条记录。

## 说明
- 单次爬取仅能获取当前播放量快照，无法获取历史播放量变化
- 播放量"随时间变化"的分析逻辑见模块3（数据分析），通过按发布时间分组聚合当前播放量实现

# 数据集格式

## 爬虫原始字段（17列）
| # | 列名 | 类型 | 说明 |
|---|------|------|------|
| 1 | `aid` | int | 视频AV号 |
| 2 | `bvid` | str | 视频BV号（与`page`组成复合唯一键，用于去重） |
| 3 | `page` | int | 分P页码（单P视频为1，多P合集为实际页码） |
| 4 | `mid` | int | UP主用户UID（唯一标识，由搜索API附带） |
| 5 | `title` | str | 视频/分P标题（已去除 `<em>` 标签） |
| 6 | `author` | str | UP主名称（上传者昵称） |
| 7 | `publish_date` | datetime | 发布时间 |
| 8 | `duration_sec` | int | 视频时长（秒） |
| 9 | `play_count` | int | 播放量（多P均分时为估算值） |
| 10 | `danmaku_count` | int | 弹幕数 |
| 11 | `comment_count` | int | 评论数 |
| 12 | `favorite_count` | int | 收藏数 |
| 13 | `coin_count` | int | 硬币数 |
| 14 | `share_count` | int | 分享数 |
| 15 | `like_count` | int | 点赞数 |
| 16 | `tags` | str | 视频标签（逗号分隔） |
| 17 | `category` | str | 视频分区（如"VOCALOID·UTAU"） |
| 18 | `url` | str | 视频链接 |
| 19 | `copyright` | int | B站版权标记（1=自制, 2=转载） |
| 20 | `play_estimated` | bool | 播放量是否为均分估算 |

## 清洗后新增衍生字段（+7列）
| # | 列名 | 类型 | 说明 |
|---|------|------|------|
| 21 | `song_name` | str | 从标题解析出的歌曲名称 |
| 22 | `original_creator` | str | 原作者（作曲/编曲/作词） |
| 23 | `vocal_singer` | str | 使用的虚拟歌姬 |
| 24 | `is_reupload` | bool | 是否为搬运作品 |
| 25 | `content_type` | str | 内容分类（见下方分类体系） |
| 26 | `days_since_pub` | int | 发布至今的天数 |
| 27 | `daily_avg_plays` | float | 日均播放量 |
| 28 | `engagement_rate` | float | 互动率 |

> 原始20列 + 衍生8列 = 共28列

## `content_type` 分类体系

| 值 | 含义 | 用途 |
|----|------|------|
| `game_cover` | 音游曲翻唱/相关（PJSK/バンドリ/Phigros/Arcaea/CHUNITHM等） | 过气分析时单独分层，避免音游热度干扰IA歌姬本身的趋势判断 |
| `vocaloid_original` | VOCALOID原创曲 | IA歌曲的核心类别 |
| `vocaloid_cover` | VOCALOID翻唱（非音游） | |
| `reupload` | 搬运/转载作品 | is_reupload=True |
| `other` | 无法分类 | 兜底 |

# 数据清洗（模块2：preprocessor.py）

预处理产出两个 DataFrame：视频粒度的 `df_clean` 和歌曲粒度的 `df_songs`。

---

## 输出一：视频粒度清洗 (df_clean)

### 1. 内容分类：QLoRA 微调本地模型（主方案）

B站视频标题格式多样、tags 混杂大量噪音（推广标签、游戏术语、飞机型号"IA58"等）。纯正则匹配对标题变体覆盖不全，无法区分"IA虚拟歌姬"和"IA58攻击机"这类语义差异。

用 30-50 条标注样本 QLoRA 微调小模型，比 few-shot 准确率更高、推理更快。

| 项 | 选择 |
|----|------|
| 基座模型 | Qwen2.5-1.5B-Instruct |
| 微调方法 | QLoRA（4-bit 量化 + LoRA, r=8, alpha=16） |
| 框架 | `unsloth` |
| 训练数据 | 30-50 条人工标注样本（JSON） |
| 推理速度 | ~5-10 条/秒 |

**分类标签**：`game_cover` | `vocaloid_original` | `vocaloid_cover` | `irrelevant` | `other`

（训练流程和推理代码详见之前版本，此处省略）

### 2. 正则分类（兜底方案）

LLM 异常时 fallback 到正则。规则同上版：音游关键词 → 搬运判定 → 原创/翻唱匹配。

### 3. 标题解析

提取 `【】`/`[]` 中标签 → `vocal_singer`；识别搬运/Cover → `is_reupload`；识别创作者 → `original_creator`；剩余文本 → `song_name`。

### 4. 通用清洗 + 衍生特征

去重 `(bvid, page)` → 缺失值填0 → 日期标准化 → 异常值过滤 → `days_since_pub` / `daily_avg_plays` / `engagement_rate`。

---

## 输出二：歌曲粒度聚合 (df_songs)

### 动机

4000+ 条视频记录中，同一首歌可能被 20 个 UP主 分别上传（官方 PV、搬运、翻唱、音游录屏）。过气分析需要以**歌曲**为基本单位——"六兆年と一夜物語 作为一首歌，关注度是否在下降？"

### 聚合键

```
(song_name, original_creator)
```

同一首歌 + 同一创作者 = 同一条歌曲记录。不同创作者的同一歌名（如翻唱版）视为不同歌曲。

### 聚合函数

```python
def aggregate_to_songs(df_clean: pd.DataFrame) -> pd.DataFrame:
    df_songs = df_clean.groupby(["song_name", "original_creator"]).agg(
        # 视频数量
        video_count=("bvid", "nunique"),

        # 播放量：总分 + 均值 + 最高
        total_plays=("play_count", "sum"),
        mean_plays=("play_count", "mean"),
        max_plays=("play_count", "max"),

        # 时间
        first_publish=("publish_date", "min"),
        last_publish=("publish_date", "max"),

        # 歌姬（取最常见的）
        vocal_singer=("vocal_singer", lambda x: x.mode().iloc[0] if len(x.mode()) > 0 else "未知"),

        # 内容分类（取主要类型，出现最多的）
        content_type=("content_type", lambda x: x.mode().iloc[0] if len(x.mode()) > 0 else "other"),

        # 互动数据总和
        total_danmaku=("danmaku_count", "sum"),
        total_comment=("comment_count", "sum"),
        total_favorite=("favorite_count", "sum"),
        total_coin=("coin_count", "sum"),
        total_share=("share_count", "sum"),
        total_like=("like_count", "sum"),

        # 平均互动率
        mean_engagement=("engagement_rate", "mean"),

        # UP主列表（去重后逗号拼接）
        uploaders=("author", lambda x: ", ".join(x.unique()[:10])),
    ).reset_index()

    # 衍生：歌曲存活天数
    df_songs["active_days"] = (df_songs["last_publish"] - df_songs["first_publish"]).dt.days

    # 衍生：日均播放（整首歌的生命周期内）
    df_songs["daily_avg_plays"] = df_songs["total_plays"] / df_songs["active_days"].clip(lower=1)

    return df_songs
```

### 聚合后字段（18列）

| # | 列名 | 类型 | 说明 |
|---|------|------|------|
| 1 | `song_name` | str | 歌曲名称（聚合键） |
| 2 | `original_creator` | str | 原作者（聚合键） |
| 3 | `video_count` | int | 关联视频数 |
| 4 | `total_plays` | int | 总播放量 |
| 5 | `mean_plays` | float | 单视频平均播放量 |
| 6 | `max_plays` | int | 最高单视频播放量 |
| 7 | `first_publish` | datetime | 最早发布时间 |
| 8 | `last_publish` | datetime | 最晚发布时间 |
| 9 | `vocal_singer` | str | 主要歌姬 |
| 10 | `content_type` | str | 主要分类 |
| 11 | `total_danmaku` | int | 总弹幕 |
| 12 | `total_comment` | int | 总评论 |
| 13 | `total_favorite` | int | 总收藏 |
| 14 | `total_coin` | int | 总硬币 |
| 15 | `total_share` | int | 总分享 |
| 16 | `total_like` | int | 总点赞 |
| 17 | `mean_engagement` | float | 平均互动率 |
| 18 | `uploaders` | str | UP主列表（前10） |
| 19 | `active_days` | int | 活跃天数 |
| 20 | `daily_avg_plays` | float | 生命周期日均播放 |

### 用途

| 分析 | 使用的字段 |
|------|-----------|
| 歌曲热度排行 | `song_name`, `total_plays` |
| P主影响力排行 | `original_creator`, `total_plays`, `video_count` |
| **过气分析** | `first_publish`, `total_plays`（按首次发布时间聚合） |
| 翻唱热度 | `video_count`, `mean_plays`（多视频=热门翻唱目标） |
| 生命周期 | `active_days`, `first_publish`, `last_publish` |

### 已知局限（第一版）

- **歌名拼写差异**："六兆年と一夜物語" vs "六兆年零一夜物语" vs "Children Record" 会被视为不同歌曲，暂时不做模糊匹配
- **同名异曲**：不同歌曲恰好同名会被错误合并，概率较低暂不处理
- 后续可用 `song_name` 做 `difflib.get_close_matches()` 或调用 LLM 进行歌名归一化

---

## 预处理完整流程

```
原始DataFrame
    ↓
去重 (bvid, page)
    ↓
QLoRA 分类 → content_type（正则兜底）
    ↓
标题解析 → song_name, original_creator, vocal_singer, is_reupload
    ↓
缺失值处理 + 日期标准化 + 异常值过滤
    ↓
衍生特征 → days_since_pub, daily_avg_plays, engagement_rate
    ↓
    ├──→ df_clean（视频粒度，28列）
    │
    └──→ aggregate_to_songs(df_clean) → df_songs（歌曲粒度，20列）
```

> 两个 DataFrame 均存入 Flask `g` 对象：`g.df_clean` 和 `g.df_songs`。分析API根据查询类型自动选择合适的粒度。

# 模块3：数据分析引擎（analyzer.py）

## 设计原则

- 所有函数接收 DataFrame 为第一参数，不绑定数据源——可传 `df_clean`（视频粒度），也可传 `df_songs`（歌曲粒度）
- `get_trend_by_date` 是核心底层：用 `df_clean` 时按 `publish_date` 聚合，用 `df_songs` 时按 `first_publish` 聚合
- P主粒度由 `df_songs` 的 `groupby("original_creator")` 按需派生，不持久化

---

## 1. 通用工具（任意 DataFrame 可用）

### 1.1 `get_summary_stats(df)`

```
输入: pd.DataFrame（任意数值列）
输出: dict
      {
        "row_count": int,
        "columns": {
          "play_count": {"count": int, "mean": float, "std": float,
                         "min": float, "25%": float, "50%": float,
                         "75%": float, "max": float},
          ...
        }
      }
功能: 对所有数值列输出 describe()，前端以表格展示数据概览
```

### 1.2 `get_top_n(df, column, n=10, ascending=False)`

```
输入: pd.DataFrame, column="total_plays", n=10
输出: pd.DataFrame（原数据子集，保留所有列，按指定列排序的前 n 行）
功能: 排行榜查询。"播放量最高的视频/歌曲/P主"
```

### 1.3 `get_correlation_matrix(df, columns)`

```
输入: pd.DataFrame, columns=["play_count","like_count","coin_count",...]
输出: pd.DataFrame（N×N 相关系数矩阵，值域 [-1,1]）
功能: 互动指标间的线性相关程度。1.0=完全正相关，-1.0=完全负相关
```

### 1.4 `filter_by_condition(df, column, op, value)`

```
输入: pd.DataFrame, column="content_type", op="==", value="game_cover"
      支持的 op: "==", "!=", ">", "<", ">=", "<=", "contains"
输出: pd.DataFrame（筛选后的子集）
功能: 动态条件过滤。前端选择器 → 此函数 → 返回过滤后数据
```

### 1.5 `compare_groups(df, group_col, metric_col)`

```
输入: pd.DataFrame, group_col="content_type", metric_col="play_count"
输出: dict
      {
        "game_cover": {
          "count": int, "mean": float, "median": float,
          "sum": int, "std": float
        },
        ...
      }
功能: 按分组列统计某指标的 mean/median/sum/std
```

---

## 2. 视频粒度分析（df_clean → "谁在发、怎么发"）

### 2.1 `get_author_stats(df, author)`

```
输入: pd.DataFrame (df_clean), author="可可san"
输出: dict
      {
        "author": str,
        "video_count": int,                # 总投稿数
        "total_plays": int,                # 总播放量
        "mean_plays": float,               # 平均每视频播放量
        "mean_engagement": float,          # 平均互动率
        "first_video": str (ISO date),     # 首次投稿时间
        "last_video": str (ISO date),      # 最近投稿时间
        "content_type_breakdown": {        # 投稿类型分布
          "vocaloid_original": int,
          "vocaloid_cover": int,
          "game_cover": int
        },
        "top_videos": [                    # 播放量最高的 5 个视频
          {"title": str, "play_count": int, "url": str}, ...
        ]
      }
功能: 某 UP主 的全景画像
```

### 2.2 `get_content_type_distribution(df)`

```
输入: pd.DataFrame (df_clean)
输出: dict
      {
        "game_cover":        {"count": int, "ratio": float},
        "vocaloid_original": {"count": int, "ratio": float},
        "vocaloid_cover":    {"count": int, "ratio": float},
        "reupload":          {"count": int, "ratio": float},
        "irrelevant":        {"count": int, "ratio": float},
        "other":             {"count": int, "ratio": float}
      }
功能: 各 content_type 的数量和占比，前端渲染为饼图
```

### 2.3 `get_tag_frequency(df, top_n=30)`

```
输入: pd.DataFrame (df_clean), top_n=30
输出: list[tuple]  [("IA", 3047), ("VOCALOID", 2091), ...]
功能: 所有 tags 的出现频次（逗号分隔后统计），用于词云或横向柱状图
```

### 2.4 `get_upload_trend(df, group_by='month')`

```
输入: pd.DataFrame (df_clean), group_by='month'|'year'
输出: pd.DataFrame  [period, video_count]
功能: 每月/年 IA 视频投稿数量变化，反映 UP主端活跃度
```

---

## 3. 歌曲粒度分析（df_songs → "什么歌火、火多久"）

### 3.1 `get_trend_by_date(df, group_by='month', filters=None)`

```
输入: pd.DataFrame (df_songs 或 df_clean),
      group_by='month'|'year',
      filters={'content_type': ['vocaloid_original','vocaloid_cover']}  # 可选
输出: pd.DataFrame
      [period, count, mean_plays, median_plays, sum_plays]
      2020-01  5  120,000  85,000  600,000
      2020-02  8   95,000  62,000  760,000
      ...
      若输入 df_songs → 时间轴 = first_publish；输入 df_clean → 时间轴 = publish_date
功能: 核心趋势函数。按时间段聚合该时段发布作品的播放量统计。
      是 get_trend_comparison / get_creator_trend / get_new_song_trend 的底层
```

### 3.2 `get_trend_comparison(df, group_by='month')`

```
输入: pd.DataFrame (df_songs), group_by='month'|'year'
输出: dict
      {
        "all":       pd.DataFrame,   # 所有歌曲
        "non_game":  pd.DataFrame,   # content_type != game_cover
        "game_only": pd.DataFrame    # content_type == game_cover
      }
功能: 三条趋势线并行对比。灌入 make_trend_comparison_chart() 生成核心可视化
```

### 3.3 `get_decline_judgment(df)`

```
输入: pd.DataFrame (df_songs) 或 get_trend_by_date 的输出
输出: dict
      {
        "early_period": str,           # "2019-01 ~ 2020-06"
        "early_mean_plays": float,
        "recent_period": str,          # "2023-07 ~ 2024-12"
        "recent_mean_plays": float,
        "ratio": float,                # recent / early
        "judgment": str,               # "显著下降" | "热度上升" | "持平/波动"
      }
判定规则: ratio < 0.5 → "显著下降" / ratio > 1.0 → "热度上升" / 否则 → "持平/波动"
功能: 量化歌曲粒度过气程度。前端展示为结论卡片
```

### 3.4 `get_song_rankings(df, metric='total_plays', n=10)`

```
输入: pd.DataFrame (df_songs), metric='total_plays'|'video_count'|'mean_plays'|'max_plays'
输出: pd.DataFrame（按 metric 降序前 n 行）
功能: "播放量最高的10首歌"、"翻唱版本最多的10首歌"
```

### 3.5 `get_lifecycle_stats(df)`

```
输入: pd.DataFrame (df_songs)
输出: dict
      {
        "mean_active_days": float,
        "median_active_days": float,
        "longest_active": {
          "song_name": str, "original_creator": str,
          "active_days": int, "video_count": int
        },
        "distribution": [
          {"range": "0-30天",  "count": int},
          {"range": "31-180天","count": int},
          {"range": "181-365天","count": int},
          {"range": "1-3年",   "count": int},
          {"range": "3年以上",  "count": int}
        ]
      }
功能: 歌曲生命周期分析。"IA歌能活多久？六兆年是不是持续有翻唱？"
```

### 3.6 `get_new_song_trend(df, group_by='year')`

```
输入: pd.DataFrame (df_songs), group_by='year'|'month'
输出: pd.DataFrame
      [period, new_song_count]
      2015  245
      2016  198
      2017  167
      ...
功能: 每年/月首次投稿的新曲数量变化。
      下行趋势 = 整体"流入"在减少 → 社区基础过气信号
```

### 3.7 `get_new_song_performance(df, recent_months=12)`

```
输入: pd.DataFrame (df_songs), recent_months=12
输出: dict
      {
        "recent_song_count": int,            # 近N月新曲数
        "recent_mean_plays": float,          # 近N月新曲平均总播放
        "recent_median_plays": float,
        "historical_mean_plays": float,      # 历史所有歌曲平均总播放
        "ratio": float,                      # recent / historical
        "judgment": str,                     # "新曲表现高于历史" | "低于历史" | "持平"
      }
功能: 近期新曲播放量 vs 历史基线。"新歌质量是否不如从前？"
      与 get_new_song_trend 互补：前者看数量，此函数看质量
```

### 3.8 `get_recent_hot_songs(df, recent_months=6, n=10)`

```
输入: pd.DataFrame (df_songs), recent_months=6, n=10
输出: pd.DataFrame（近N月发布，按 total_plays 降序前 n 行）
功能: 新曲推荐。"最近半年最火的新IA曲是哪些？"
```

---

## 4. P主粒度分析（df_songs → groupby original_creator）

### 4.1 `get_creator_rankings(df, metric='total_plays', n=10)`

```
输入: pd.DataFrame (df_songs), metric='total_plays'|'song_count'
输出: pd.DataFrame
      [original_creator, song_count, total_plays,
       mean_plays_per_song, first_song, last_song]
功能: P主总播放/作品数排行。"哪个P主的IA曲总播放最高？"
```

### 4.2 `get_creator_trend(df, creator_name, group_by='year')`

```
输入: pd.DataFrame (df_songs), creator_name="Orangestar", group_by='year'|'month'
输出: pd.DataFrame
      [period, song_count, mean_total_plays]
      2015  2  3,200,000
      2016  1  1,500,000
      ...
功能: 某P主各年新曲数量和平均总播放量。双线图：柱状=新曲数，折线=平均播放
```

### 4.3 `get_creator_comparison(df, creator_names, group_by='year')`

```
输入: pd.DataFrame (df_songs),
      creator_names=["Orangestar","jin","kemu"], group_by='year'
输出: dict
      {
        "Orangestar": pd.DataFrame,   # 同 get_creator_trend 输出
        "jin":         pd.DataFrame,
        "kemu":        pd.DataFrame
      }
功能: 多位 P主 趋势并排对比，前端渲染为多系列折线图
```

### 4.4 `get_creator_activity(df, creator_name)`

```
输入: pd.DataFrame (df_songs), creator_name="Orangestar"
输出: dict
      {
        "creator_name": str,
        "total_songs": int,
        "active_years": int,                # 有投稿的年份数
        "last_new_song": str (ISO date),    # 最近一次新曲发布时间
        "months_since_last": int,           # 距最新投稿多少个月
        "peak_year": str,                   # 新曲最多的年份
        "peak_year_count": int,
        "status": str,                      # "活跃" | "停更中" | "已沉寂"
      }
判定规则: months_since_last ≤ 12 → "活跃" / ≤ 36 → "停更中" / > 36 → "已沉寂"
功能: P主活跃度判定。"Orangestar 还在写IA曲吗？上次发新歌是什么时候？"
```

---

## 5. 完整函数清单

| # | 函数 | 粒度 | 用途 |
|---|------|:--:|------|
| 1.1 | `get_summary_stats` | 通用 | 数值概览 |
| 1.2 | `get_top_n` | 通用 | 排行榜 |
| 1.3 | `get_correlation_matrix` | 通用 | 互动相关性 |
| 1.4 | `filter_by_condition` | 通用 | 动态筛选 |
| 1.5 | `compare_groups` | 通用 | 分组对比 |
| 2.1 | `get_author_stats` | 视频 | UP主画像 |
| 2.2 | `get_content_type_distribution` | 视频 | 类型分布 |
| 2.3 | `get_tag_frequency` | 视频 | 标签频次 |
| 2.4 | `get_upload_trend` | 视频 | 投稿趋势 |
| 3.1 | `get_trend_by_date` | 歌曲 | **时间聚合（底层）** |
| 3.2 | `get_trend_comparison` | 歌曲 | 分层趋势对比 |
| 3.3 | `get_decline_judgment` | 歌曲 | 过气判定 |
| 3.4 | `get_song_rankings` | 歌曲 | 歌曲排行 |
| 3.5 | `get_lifecycle_stats` | 歌曲 | 生命周期 |
| 3.6 | `get_new_song_trend` | 歌曲 | **新曲数量趋势** |
| 3.7 | `get_new_song_performance` | 歌曲 | **新曲质量对比** |
| 3.8 | `get_recent_hot_songs` | 歌曲 | **新曲推荐** |
| 4.1 | `get_creator_rankings` | P主 | P主排行 |
| 4.2 | `get_creator_trend` | P主 | 单P主趋势 |
| 4.3 | `get_creator_comparison` | P主 | 多P主对比 |
| 4.4 | `get_creator_activity` | P主 | **P主活跃度** |

> 共 21 个函数。加粗为本次新增。
> 过气分析 = 存量维度（老歌播放量 3.1/3.2/3.3）+ 流量维度（新曲数量 3.6 + 新曲表现 3.7）+ P主活跃度 4.4。

# 模块4：数据可视化（visualizer.py）

## 设计思路
不直接生成图片，而是生成 **ECharts option JSON** 返回给前端，由 `main.js` 调用 `echarts.setOption()` 渲染。图表具有完整的交互能力（缩放、hover详情、图例切换、数据点选中等）。

所有函数输出 `dict`（即 ECharts option 对象），可直接 `json.dumps()` 后传给前端。

---

## 1. 基础图表

### 1.1 `make_bar_chart(labels, values, title, x_name, y_name, horizontal=False)`

```
输入: labels=list[str], values=list[int/float], title="", x_name="", y_name=""
输出: dict (ECharts option) — 柱状图（horizontal=True 则为横向柱状图）
用途: 通用柱状图。排行榜、分组对比均可调用
```

### 1.2 `make_line_chart(x_data, y_series, title, x_name, y_name)`

```
输入: x_data=list[str], y_series=dict {"系列名": [值列表], ...}, title=""
输出: dict — 单/多系列折线图
用途: 通用时间趋势图。投稿趋势、歌曲热度趋势、新曲数量趋势
```

### 1.3 `make_pie_chart(labels, values, title)`

```
输入: labels=list[str], values=list[int], title=""
输出: dict — 饼图/环形图
用途: content_type 分布占比、版权分布（自制vs转载）
```

### 1.4 `make_scatter_chart(x_values, y_values, labels, title, x_name, y_name)`

```
输入: x_values=list[float], y_values=list[float], labels=list[str]（悬浮标签）
输出: dict — 散点图
用途: 播放量 vs 互动率，每个点代表一首歌/一个视频
      可扩展 color_values 参数按 content_type 着色
```

### 1.5 `make_heatmap_data(corr_df)`

```
输入: pd.DataFrame（相关系数矩阵）
输出: dict — ECharts heatmap option
用途: 互动指标相关性热力图（play/coin/like/fav/comment/danmaku）
```

---

## 2. 排行榜与对比图表

### 2.1 `make_ranking_bar(df, col, top_n, title)`

```
输入: df=pd.DataFrame, col=str（排名依据列）, top_n=int, title=""
输出: dict — 横向柱状图（自动取 top_n 并排序）
用途: "播放量 TOP15 UP主"、"总播放 TOP10 歌曲"、"标签频次 TOP30"
```

### 2.2 `make_grouped_bar(df, group_col, metric_col, title)`

```
输入: df=pd.DataFrame, group_col="content_type", metric_col="mean_plays"
输出: dict — 分组柱状图（每组一根柱）
用途: 各 content_type 的平均播放量对比、原创vs搬运vs音游 的播放量中位数对比
      → 配合 analyzer.compare_groups() 的结果使用
```

### 2.3 `make_comparison_bar(compare_dict, title)`

```
输入: dict
      {
        "labels": ["近期新曲", "历史基线"],
        "values": [52000, 85000],
        "metric_name": "平均播放量"
      }
输出: dict — 双柱对比图，X轴=两个类别，Y轴=指标值
用途: 近期新曲播放量 vs 历史基线、某P主 vs 全体P主的平均播放
      → 配合 analyzer.get_new_song_performance() 的结果使用
```

---

## 3. 趋势分析图表

### 3.1 `make_trend_comparison_chart(comparison, metric, title)`

```
输入: comparison=dict（analyzer.get_trend_comparison() 的输出）,
      metric='mean_plays'|'median_plays'|'sum_plays'|'count',
      title=""
输出: dict — 三系列折线图
      全部 (灰色虚线) | 非音游 (蓝色实线) | 仅音游 (橙色实线)
用途: 过气分析核心图。三条线放一张图，图例切换对比
```

### 3.2 `make_dual_axis_chart(x_data, bar_series, line_series, title, bar_name, line_name)`

```
输入: x_data=list[str]（时间轴，如年份列表）,
      bar_series=list[int]（柱状数据，如新曲数量）,
      line_series=list[float]（折线数据，如平均播放量）,
      title="", bar_name="新曲数", line_name="平均播放"
输出: dict — 双Y轴图表（左轴=柱状，右轴=折线）
用途: P主趋势分析。同时展示"每年发了多少歌"和"这些歌的平均播放量"
      → 配合 analyzer.get_creator_trend() 的结果使用
```

### 3.3 `make_multi_creator_line(trends_dict, title, metric)`

```
输入: trends_dict=dict（analyzer.get_creator_comparison() 的输出）,
      title="", metric="mean_total_plays"
输出: dict — 多系列折线图（每位P主一条线，颜色自动分配）
用途: 多位P主趋势并排对比。"jin vs kemu vs Orangestar"
      → 配合 analyzer.get_creator_comparison() 的结果使用
```

---

## 4. 分布图表

### 4.1 `make_histogram(values, bins, title, x_name)`

```
输入: values=list[float/int]（原始数据，如 active_days 列表）,
      bins=int（分箱数）,
      title="", x_name="活跃天数"
输出: dict — 直方图（自动计算频次和区间）
用途: 活跃天数分布、播放量分布、投稿数分布
      → 配合 analyzer.get_lifecycle_stats() 的 distribution 使用
```

---

## 5. 图表与分析的对应关系

| 图表函数 | 配合的分析函数 | 图表类型 |
|---------|--------------|------|
| `make_bar_chart` | 1.2 `get_top_n`, 1.5 `compare_groups` | 柱状图 |
| `make_line_chart` | 2.4 `get_upload_trend`, 3.1 `get_trend_by_date`, 3.6 `get_new_song_trend` | 折线图 |
| `make_pie_chart` | 2.2 `get_content_type_distribution` | 饼图 |
| `make_scatter_chart` | 视频/歌曲粒度的任意双变量关系 | 散点图 |
| `make_heatmap_data` | 1.3 `get_correlation_matrix` | 热力图 |
| `make_ranking_bar` | 1.2 `get_top_n`, 2.3 `get_tag_frequency`, 3.4 `get_song_rankings`, 3.8 `get_recent_hot_songs`, 4.1 `get_creator_rankings` | 横向柱状图 |
| `make_grouped_bar` | 1.5 `compare_groups` (多指标), 2.2 `get_content_type_distribution` (对比版) | 分组柱状图 |
| `make_comparison_bar` | 3.7 `get_new_song_performance` | 双柱对比图 |
| `make_trend_comparison_chart` | 3.2 `get_trend_comparison` | 三线对比折线图 |
| `make_dual_axis_chart` | 4.2 `get_creator_trend` | 双Y轴图 |
| `make_multi_creator_line` | 4.3 `get_creator_comparison` | 多P主对比折线图 |
| `make_histogram` | 3.5 `get_lifecycle_stats` | 直方图 |

> 所有 21 个分析函数中，5 个输出为纯文本/表格/卡片（1.1, 1.4, 2.1, 3.3, 4.4），其余 16 个均有对应图表函数覆盖。

---

## 6. 返回格式示例

### 折线图 (make_line_chart)
```json
{
  "title": {"text": "IA新曲数量趋势", "left": "center"},
  "tooltip": {"trigger": "axis"},
  "xAxis": {"type": "category", "data": ["2015", "2016", ...]},
  "yAxis": {"type": "value", "name": "新曲数"},
  "series": [{"name": "新曲数", "type": "line", "data": [245, 198, ...], "smooth": true}]
}
```

### 双Y轴图 (make_dual_axis_chart)
```json
{
  "title": {"text": "Orangestar IA作品趋势", "left": "center"},
  "tooltip": {"trigger": "axis"},
  "legend": {"data": ["新曲数", "平均播放量"]},
  "xAxis": {"type": "category", "data": ["2015", "2016", ...]},
  "yAxis": [
    {"type": "value", "name": "新曲数"},
    {"type": "value", "name": "平均播放量"}
  ],
  "series": [
    {"name": "新曲数", "type": "bar", "data": [2, 1, ...]},
    {"name": "平均播放量", "type": "line", "yAxisIndex": 1, "data": [3200000, 1500000, ...]}
  ]
}
```

### 分组柱状图 (make_grouped_bar)
```json
{
  "title": {"text": "各内容类型播放量对比", "left": "center"},
  "tooltip": {"trigger": "axis"},
  "legend": {"data": ["平均播放量", "中位播放量"]},
  "xAxis": {"type": "category", "data": ["game_cover", "vocaloid_original", "vocaloid_cover"]},
  "yAxis": {"type": "value"},
  "series": [
    {"name": "平均播放量", "type": "bar", "data": [85000, 120000, 95000]},
    {"name": "中位播放量", "type": "bar", "data": [15000, 45000, 22000]}
  ]
}
```

# 模块5：AI问答助手（ai_assistant.py）

## 设计思路

用户在前端输入自然语言问题，系统通过 **DeepSeek V4 Flash + Function Calling** 自动选择预设分析函数，执行后返回结果 + 图表 + 自然语言解读。

### 为什么不生成代码？

生成代码方案（`exec()` 执行模型输出的 pandas）有安全风险、输出格式不可控。Function Calling 让模型只负责**选函数 + 传参**，实际计算由 analyzer/visualizer 完成，安全可靠。

---

## 交互流程

```
用户输入问题
    ↓
ai_assistant 构造 System Prompt + 注册 Tools（分析/图表函数）
    ↓
DeepSeek V4 Flash → 返回 tool_calls 或 文本回答
    ↓
    ├── 有 tool_calls → 后端调用对应函数 → 将结果注入对话 → 再次请求 DeepSeek
    │                                                      ↓
    │                                   模型判断是否需要图表 → 调用 visualizer
    │                                                      ↓
    └── 纯文本回答 ←────────────────────────────── 最终回复 + 图表JSON + 解读
```

### 多轮调用

一次复杂问题可能需要 2-3 轮：

```
轮1: 用户提问 → DeepSeek 决定调用 get_creator_trend()
轮2: 函数结果返回 → DeepSeek 解读数据，决定是否需要 make_dual_axis_chart()
轮3: 图表生成 → DeepSeek 输出最终回复（文本解读 + 图表JSON）
```

---

## 暴露给 AI 的 Tools（15 个）

### 数据查询（4 个）

| Tool 名 | 说明 |
|---------|------|
| `get_summary_stats` | 数据概览（行数、数值列的均值/中位数/分布） |
| `get_top_n` | 排行榜查询 |
| `filter_by_condition` | 按条件筛选数据 |
| `compare_groups` | 分组对比统计 |

### 视频粒度分析（2 个）

| Tool 名 | 说明 |
|---------|------|
| `get_author_stats` | UP主画像（总投稿、总播放、类型分布、TOP5视频） |
| `get_upload_trend` | 投稿数量时间趋势 |

### 歌曲粒度分析（6 个）

| Tool 名 | 说明 |
|---------|------|
| `get_trend_comparison` | 三线分层趋势对比（存量过气分析） |
| `get_decline_judgment` | 过气量化判定 |
| `get_song_rankings` | 歌曲热度/翻唱排行 |
| `get_lifecycle_stats` | 歌曲生命周期统计 |
| `get_new_song_trend` | 新曲数量趋势（流量过气分析） |
| `get_new_song_performance` | 新曲质量 vs 历史基线 |

### P主粒度分析（2 个）

| Tool 名 | 说明 |
|---------|------|
| `get_creator_activity` | P主活跃度判定 |
| `get_creator_rankings` | P主影响力排行 |

### 图表生成（1 个）

| Tool 名 | 说明 |
|---------|------|
| `generate_chart` | 根据分析结果 + chart_type 生成 ECharts option JSON |

> 图表函数（12 个 visualizer）不逐一暴露。`generate_chart` 接收分析结果和图表类型字符串，内部路由到对应 visualizer 函数。

### 不暴露的函数（原因）

| 函数 | 原因 |
|------|------|
| `get_trend_by_date` | 底层函数，AI 通过 `get_trend_comparison` 间接使用 |
| `get_creator_trend` | AI 通过 `get_creator_comparison` + `get_creator_activity` 覆盖 |
| `get_creator_comparison` | 参数复杂（P主名列表），改为 natural language → `get_creator_activity` 循环调用 |
| `get_content_type_distribution` | 预设图表，非问答场景 |
| `get_tag_frequency` | 同上 |
| `get_recent_hot_songs` | 同上 |
| `get_correlation_matrix` | 同上 |

---

## System Prompt

```python
SYSTEM_PROMPT = """你是一个B站IA虚拟歌姬数据分析助手，帮助用户分析IA（-ARIA ON THE PLANETES-）音乐作品的数据。

## 你的能力
- 查询数据概览、排行榜、趋势
- 判断IA歌曲或某位P主（创作者）是否"过气"
- 对比不同歌曲类型（原创/翻唱/音游曲）的表现
- 分析新曲趋势和P主活跃度
- 生成交互式图表（折线图、柱状图、饼图等）

## 数据说明
系统中有两个数据集：
1. 视频粒度（df_clean）：每条记录是一个B站视频投稿，包含UP主、标题、播放量、标签等
2. 歌曲粒度（df_songs）：每条记录是一首歌（同一首歌的多个视频已聚合），包含总播放量、视频数、首次发布时间等

## 重要概念
- content_type 分类：game_cover(音游曲)、vocaloid_original(VOCALOID原创)、vocaloid_cover(VOCALOID翻唱)、irrelevant(无关混入内容)、other
- original_creator：歌曲的原作者/P主（作曲/编曲），非UP主
- "过气"判断看两个维度：存量（老歌播放量是否停滞）+ 流量（新曲数量和质量是否下降）+ P主是否还在创作
- play_estimated=True 表示播放量是从多P合集均分估算的，分析时应谨慎对待

## 回答风格
- 先给出数据结论，再展示图表，最后简要解读
- 用中文回答，数据用千分位格式（如 1,234,567）
- 如果数据不足以回答，直接说明限制
- 涉及过气判断时，同时给出存量和流量维度的证据
- 对比分析时优先用图表而非纯文字

## 可用函数
你可以调用以下函数获取数据。如果需要生成图表，调用 generate_chart 并传入 chart_type 参数。

可用的 chart_type：
- "line"：折线图（趋势数据）
- "bar"：柱状图（排行榜、对比）
- "pie"：饼图（分布占比）
- "dual_axis"：双Y轴图（P主趋势：柱=新曲数，折线=平均播放）
- "histogram"：直方图（分布）
- "comparison_bar"：双柱对比图（近期 vs 历史）
- "trend_comparison"：三线对比图（全部/非音游/仅音游）
- "heatmap"：热力图（相关性矩阵）
- "scatter"：散点图（双变量关系）

## 常见问题示例
Q: "IA最火的10首歌是什么？"
A: 调用 get_song_rankings(metric="total_plays", n=10)，用 bar 图表展示

Q: "Orangestar 还在写IA曲吗？他过气了吗？"
A: 调用 get_creator_activity("Orangestar") 和 get_decline_judgment()，
   如有数据再调用 generate_chart(chart_type="dual_axis")

Q: "IA的新曲是不是越来越少了？"
A: 调用 get_new_song_trend() 和 get_new_song_performance()，
   用 line 图表展示新曲数量趋势

Q: "原创曲和翻唱哪个更火？"
A: 调用 compare_groups(group_col="content_type", metric_col="play_count")，
   用 grouped_bar 图表展示

Q: "最近有哪些值得关注的新IA曲？"
A: 调用 get_recent_hot_songs(recent_months=6, n=10)

## 禁令
- 不要编造数据。所有数据必须通过函数调用获取。
- 不要猜测图表需求。需要图表时调用 generate_chart。
- 不确定时先调用 get_summary_stats 了解数据概况。
- 如果一个函数无法回答，尝试组合多个函数。
"""
```

---

## 核心代码结构

```python
import json
from openai import OpenAI

client = OpenAI(
    api_key="your-deepseek-api-key",
    base_url="https://api.deepseek.com",
)

TOOLS = [
    {
        "type": "function",
        "function": {
            "name": "get_summary_stats",
            "description": "获取当前数据集的数值概览（行数、均值、中位数等）",
            "parameters": {
                "type": "object",
                "properties": {},
                "required": []
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "get_top_n",
            "description": "获取按指定列排序的前N条记录（排行榜）",
            "parameters": {
                "type": "object",
                "properties": {
                    "column": {"type": "string", "description": "排序依据列，如 total_plays, play_count, video_count"},
                    "n": {"type": "integer", "description": "返回条数，默认10"},
                    "granularity": {"type": "string", "enum": ["video", "song", "creator"], "description": "数据粒度"}
                },
                "required": ["column"]
            }
        }
    },
    # ... 其余 13 个 tools
    {
        "type": "function",
        "function": {
            "name": "generate_chart",
            "description": "根据分析结果生成 ECharts 图表。需要提供 chart_type",
            "parameters": {
                "type": "object",
                "properties": {
                    "chart_type": {
                        "type": "string",
                        "enum": ["line", "bar", "pie", "dual_axis", "histogram",
                                 "comparison_bar", "trend_comparison", "heatmap", "scatter"],
                        "description": "图表类型"
                    },
                    "data": {"type": "object", "description": "分析函数返回的结果数据"},
                    "title": {"type": "string", "description": "图表标题"}
                },
                "required": ["chart_type", "data", "title"]
            }
        }
    }
]

def ask_deepseek(question: str, conversation_history: list) -> dict:
    """发送对话请求，返回 model 的 tool_calls 或 文本回复"""
    messages = [{"role": "system", "content": SYSTEM_PROMPT}]
    messages.extend(conversation_history)
    messages.append({"role": "user", "content": question})

    response = client.chat.completions.create(
        model="deepseek-chat",
        messages=messages,
        tools=TOOLS,
        tool_choice="auto",  # 模型自行判断是否需要调用函数
    )
    return response.choices[0].message


def execute_tool_call(tool_name: str, arguments: dict) -> dict:
    """根据 tool_name 路由到对应的 analyzer/visualizer 函数"""
    # 数据查询
    if tool_name == "get_summary_stats":
        return get_summary_stats(g.df_clean)
    elif tool_name == "get_top_n":
        granularity = arguments.get("granularity", "song")
        df = {"video": g.df_clean, "song": g.df_songs, "creator": g.df_songs}[granularity]
        return get_top_n(df, arguments["column"], arguments.get("n", 10))
    # ... 其余路由
    elif tool_name == "generate_chart":
        return generate_chart(arguments["chart_type"], arguments["data"], arguments["title"])
    else:
        return {"error": f"Unknown tool: {tool_name}"}


def chat_endpoint(question: str, history: list) -> dict:
    """Flask 路由 /api/chat 的处理函数"""
    # 轮1：发送问题
    msg = ask_deepseek(question, history)

    # 如果有函数调用 → 执行 → 注入结果 → 再问
    while msg.tool_calls:
        history.append({"role": "assistant", "tool_calls": msg.tool_calls})

        for tc in msg.tool_calls:
            result = execute_tool_call(tc.function.name, json.loads(tc.function.arguments))
            history.append({
                "role": "tool",
                "tool_call_id": tc.id,
                "content": json.dumps(result, ensure_ascii=False, default=str)
            })

        msg = ask_deepseek("请根据以上数据回答用户的问题。", history)

    # 返回最终回复
    return {
        "reply": msg.content,
        "has_chart": any("chart" in h.get("content", "") for h in history if h["role"] == "tool"),
        "history": history
    }
```

---

## Token 预算

| 组件 | 估算 |
|------|------|
| System Prompt | ~1200 tokens |
| Conversation History（含函数结果） | ~500-1500 tokens |
| User Question | ~100 tokens |
| Model Response | ~300-800 tokens |
| **每轮合计** | **~2000-3500 tokens** |

> 一次复杂问题（2-3 轮函数调用）总共约 4000-8000 tokens。DeepSeek V4 Flash 定价极低（输入 ¥1/百万tokens），成本可忽略。

# API 路由设计（app.py）

## 路由表
| 方法 | 路由 | 功能 | 对应模块 |
|------|------|------|----------|
| GET | `/` | 首页：上传页面 | - |
| POST | `/upload` | 接收上传文件，返回数据预览JSON | 模块1 |
| GET | `/analysis` | 分析页面 | - |
| POST | `/api/preview` | 获取数据前N行（含列信息） | 模块1 |
| POST | `/api/clean` | 执行数据清洗，返回清洗报告 | 模块2 |
| POST | `/api/describe` | 获取描述统计 | 模块3 |
| POST | `/api/top` | 获取TOP N排行（column, n可指定） | 模块3 |
| POST | `/api/correlation` | 获取相关性矩阵 | 模块3 |
| POST | `/api/trend` | 获取时间趋势数据 | 模块3 |
| POST | `/api/chart/<chart_type>` | 获取指定图表的ECharts option JSON | 模块4 |
| GET | `/chat` | AI问答页面 | - |
| POST | `/api/chat` | 发送问题，返回AI回答+可选图表 | 模块5 |

## Session 管理
使用 Flask `session` 存储当前用户的数据状态：
```python
session['data_loaded'] = True/False      # 是否已加载数据
session['data_rows'] = int               # 当前数据行数
session['columns'] = [str, ...]          # 列名列表
session['clean_done'] = True/False       # 是否已清洗
```

> 实际 DataFrame 存储在服务端全局变量或 `g` 对象中（session 无法存大对象），用户每次操作通过 API 传递操作参数，后端操作内存中的 DataFrame。

## 数据存储策略
- 用户上传文件 → 存入 `uploads/` 目录 → `file_reader` 读取到内存 → DataFrame 存入 `g.df_raw`
- 清洗后 → `g.df_clean`
- 后续分析/可视化均基于 `g.df_clean`
- 页面刷新后数据丢失，需重新上传（或使用"加载内置数据集"选项恢复）

# 前端页面设计

## 页面清单（3个独立页面 + 1个公共布局）

### base.html — 公共布局
- 顶部导航栏（Bootstrap 5 Navbar）
  - 品牌Logo：IA Music Analyzer
  - 导航链接：首页 | 数据分析 | AI问答
  - 数据状态指示器（已加载行数、清洗状态）
- 页脚：项目信息

### index.html — 首页：上传与预览
- 功能区块：
  1. 格式选择区：使用内置数据集 / 上传文件
  2. 文件上传区（拖拽上传框 + 格式说明）：支持 CSV、Excel (.xlsx/.xls)、JSON
  3. 数据预览表格（前20行，可横向滚动）
  4. 数据概览卡片（行数、列数、缺失值统计、内存占用）
  5. "前往分析"按钮 → 跳转 `/analysis`

### analysis.html — 分析页：清洗 + 统计 + 可视化
- 左侧面板（操作面板，~30%宽度）：
  - 数据清洗按钮 + 清洗报告展开区
  - 分析工具列表（描述统计、相关性、趋势、排行榜）
  - 图表选择器（下拉菜单选择图表类型）
  - 筛选条件输入（作者名、时间范围等）
- 右侧面板（结果展示，~70%宽度）：
  - 表格/统计结果区
  - ECharts 图表渲染区（自适应容器）
  - 图表导出按钮（保存为PNG）

### chat.html — AI问答页
- 聊天界面风格（类似ChatGPT）：
  - 消息列表区（用户消息靠右，AI回复靠左，含图表嵌入）
  - 底部输入框 + 发送按钮
  - 上下文提示（当前数据集概览卡片）
  - 预设问题快捷按钮（"播放量最高的是哪首？"、"原创和搬运哪个播放多？"等）

## 前端技术选型
- **Bootstrap 5**：响应式布局，组件丰富，CDN引入
- **ECharts 5**：图表渲染，CDN引入
- **原生 JavaScript**：AJAX 请求（fetch API）、DOM 操作、ECharts 初始化
- 不引入前端框架（React/Vue），保持简洁，符合"少量前端语言"要求

# 技术栈与依赖项

## Python 依赖（requirements.txt）
```
flask==3.1.*              # Web框架
pandas==2.2.*             # 数据处理核心
openpyxl==3.1.*           # Excel文件读取支持
openai==1.*               # DeepSeek API调用（兼容OpenAI SDK）
requests==2.31.*          # 爬虫HTTP请求
python-dotenv==1.0.*      # 环境变量管理（API Key）
# QLoRA 微调与推理
unsloth                   # QLoRA 训练封装（含 peft + bitsandbytes）
datasets                  # HuggingFace 数据集加载
accelerate                # 分布式训练加速
```

> `unsloth` 会自动安装 `peft`、`bitsandbytes`、`transformers`、`trl` 等依赖。
> Mac 用户需额外 `pip install torch`（MPS 后端）。

## 前端依赖（CDN引入，无需npm）
- Bootstrap 5 CSS + JS（CDN：jsDelivr）
- ECharts 5 JS（CDN：jsDelivr）

## 外部服务
- DeepSeek V4 Flash API（模块5 AI问答）
- Ollama（可选，用于对比实验或兜底）

## 开发环境
- Python 3.10+
- 虚拟环境：`python -m venv venv`
- 运行方式：`flask run` 或 `python app.py`

## 项目规模估算
| 文件 | 估计代码行数 | 来源比例 |
|------|------------|---------|
| app.py | ~80行 | 学生+AI |
| config.py | ~20行 | 学生 |
| modules/file_reader.py | ~60行 | 学生+AI |
| modules/preprocessor.py | ~200行 | 学生+AI |
| modules/analyzer.py | ~120行 | 学生+AI |
| modules/visualizer.py | ~120行 | AI+学生 |
| modules/ai_assistant.py | ~100行 | AI+学生 |
| modules/content_classifier.py | ~80行 | AI+学生 |
| train_classifier.py | ~60行 | AI+学生 |
| templates/*.html | ~300行 | AI+学生 |
| static/js/main.js | ~200行 | AI+学生 |
| static/css/style.css | ~80行 | AI |
| crawler/bilibili_crawler.py | ~280行 | AI+学生 |
| **总计** | **~1700行** | - |